# 02 · Fine-tune BERT and evaluate it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baz08/crypto-sentiment-bert/blob/main/notebooks/02_train_evaluate.ipynb)

Fine-tunes `bert-base-uncased` into a 3-class sentiment classifier (negative / neutral / positive) for crypto
Reddit text, and checks whether it beats two simpler baselines:

1. **Majority class**: always predict the most common label.
2. **TF-IDF + logistic regression**: a classic bag-of-words model.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Training takes a few minutes on a T4. On a CPU it
works but takes much longer. The optional hyperparameter search (section 4) adds about 30–40 minutes on a T4.

The code matches `deployment/bert_training/berttest.py`; this notebook adds plots and error analysis.

In [ ]:
# transformers v5 dropped TensorFlow and changed tokenizers; stay on 4.x so the
# saved model loads in the API (which pins transformers 4.46.3).
!pip install -q "transformers>=4.46,<5"

In [ ]:
# Get the repo (for the labeled data). Change BRANCH to test an unmerged branch.
REPO_URL = "https://github.com/baz08/crypto-sentiment-bert"
BRANCH = "main"

import os
if not os.path.exists("crypto-sentiment-bert"):
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL}
REPO_DIR = "crypto-sentiment-bert"
DATA_DIR = f"{REPO_DIR}/deployment/reddit"

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from sklearn import metrics
from sklearn.model_selection import train_test_split

CONFIG = {
    "base_model": "bert-base-uncased",
    "max_length": 256,   # tokens; same limit the API uses
    "epochs": 2,
    "batch_size": 16,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "test_size": 0.2,
    "seed": 42,
}
LABELS = ["Negative", "Neutral", "Positive"]

transformers.set_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, transformers {transformers.__version__}, device: {device}")
if device == "cpu":
    print("No GPU found: Runtime -> Change runtime type -> T4 GPU (training on CPU will be slow).")

## 1. Load data and split

A **shuffled, label-stratified** 80/20 split: the test set has the same label mix as the full data. (The original
notebook held out the last 500 rows, which all came from one source and were ~75% neutral.)

In [ ]:
data = pd.read_csv(f"{DATA_DIR}/Crypto_c.csv")

train_df, test_df = train_test_split(
    data, test_size=CONFIG["test_size"], random_state=CONFIG["seed"], shuffle=True, stratify=data["LABEL_COLUMN"]
)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)

pd.DataFrame({
    "train": train_df["LABEL_COLUMN"].map(dict(enumerate(LABELS))).value_counts(),
    "test": test_df["LABEL_COLUMN"].map(dict(enumerate(LABELS))).value_counts(),
}).loc[LABELS]

## 2. Baselines

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    return {
        "accuracy": metrics.accuracy_score(y_true, y_pred),
        "macro_f1": metrics.f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0),
        "per_class_f1": dict(zip(LABELS, metrics.f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0).tolist())),
        "confusion_matrix": metrics.confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist(),
    }

y_train, y_test = train_df["LABEL_COLUMN"], test_df["LABEL_COLUMN"]
results = {}

# 1) majority class
majority = y_train.mode()[0]
results["majority_baseline"] = compute_metrics(y_test, [majority] * len(y_test))

# 2) TF-IDF + logistic regression
tfidf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=CONFIG["seed"]),
)
tfidf.fit(train_df["DATA_COLUMN"], y_train)
tfidf_pred = tfidf.predict(test_df["DATA_COLUMN"])
results["tfidf_logreg"] = compute_metrics(y_test, tfidf_pred)

for name, r in results.items():
    print(f"{name:18s} accuracy {r['accuracy']:.3f}   macro-F1 {r['macro_f1']:.3f}")

## 3. Fine-tuning function

One function trains a fresh model from a config. The hyperparameter search (section 4) and the final model
(section 5) both use it, so they train in exactly the same way.

Standard fine-tuning recipe: AdamW with a small learning rate, linear warmup and decay, and gradient clipping, so
the pretrained weights aren't wrecked in the first few steps.

In [ ]:
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        CONFIG["base_model"],
        num_labels=len(LABELS),
        id2label=dict(enumerate(LABELS)),
        label2id={label: i for i, label in enumerate(LABELS)},
    ).to(device)

def make_loader(texts, labels, batch_size, shuffle):
    enc = tokenizer(list(texts), truncation=True, max_length=CONFIG["max_length"])
    items = [{**{k: v[i] for k, v in enc.items()}, "labels": int(y)} for i, y in enumerate(labels)]
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=DataCollatorWithPadding(tokenizer),
                      generator=torch.Generator().manual_seed(CONFIG["seed"]))

def predict(model, texts, batch_size=64):
    model.eval()
    texts, preds = list(texts), []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            enc = tokenizer(texts[start:start + batch_size], truncation=True, max_length=CONFIG["max_length"],
                            padding=True, return_tensors="pt").to(device)
            preds.extend(model(**enc).logits.argmax(dim=-1).cpu().tolist())
    return preds

def fine_tune(train_texts, train_labels, config, eval_texts=None, eval_labels=None, verbose=True):
    """Train a fresh model with `config`. If eval data is given, score it after every epoch.

    Returns (model, history), with one dict per epoch in history.
    """
    transformers.set_seed(CONFIG["seed"])
    model = build_model()
    loader = make_loader(train_texts, train_labels, int(config["batch_size"]), shuffle=True)
    total_steps = len(loader) * int(config["epochs"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config["lr"]), eps=1e-8,
                                  weight_decay=float(config["weight_decay"]))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(float(config["warmup_ratio"]) * total_steps), total_steps)

    history = []
    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        running_loss = 0.0
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running_loss += loss.item()

        record = {"epoch": epoch, "train_loss": running_loss / len(loader)}
        if eval_texts is not None:
            pred = predict(model, eval_texts)
            record["accuracy"] = metrics.accuracy_score(eval_labels, pred)
            record["macro_f1"] = metrics.f1_score(eval_labels, pred, average="macro")
        history.append(record)
        if verbose:
            print({k: round(v, 4) if isinstance(v, float) else v for k, v in record.items()})
    return model, history

## 4. (Optional) Hyperparameter search

`search_hyperparameters()` tries every combination in `SEARCH_GRID` and **returns the best config**:

- It holds out 15% of the *training* data as a validation set. The test set is never used for tuning, so the
  final test score stays honest.
- Each combination trains for `MAX_EPOCHS` and is scored (macro-F1) after every epoch, so the number of epochs
  is searched at no extra cost. Ties go to fewer epochs.
- Set `N_TRIALS` to try a random subset of the grid instead of all of it.

The default grid is 4 learning rates × 2 batch sizes = 8 trials × 4 epochs, about **30–40 minutes on a T4**.
Set `RUN_SEARCH = True` to run it; otherwise the final model uses the config from the setup cell.

In [ ]:
import gc
import itertools
import random

RUN_SEARCH = False
SEARCH_GRID = {"lr": [1e-5, 2e-5, 3e-5, 5e-5], "batch_size": [16, 32]}
MAX_EPOCHS = 4
N_TRIALS = None          # e.g. 4 to try a random half of the grid (about half the time)

def expand_grid(grid, n_trials=None, seed=CONFIG["seed"]):
    keys = list(grid)
    combos = [dict(zip(keys, values)) for values in itertools.product(*(grid[k] for k in keys))]
    if n_trials is not None and n_trials < len(combos):
        combos = random.Random(seed).sample(combos, n_trials)
    return combos

def search_hyperparameters(train_df, grid=SEARCH_GRID, max_epochs=MAX_EPOCHS, val_size=0.15,
                           n_trials=None, metric="macro_f1"):
    """Return (best_config, results).

    best_config: the winning hyperparameters (including epochs), found on a validation split of train_df.
    results: one row per (combination, epoch), best first.
    """
    if "epochs" in grid:
        raise ValueError("Don't put 'epochs' in the grid; set max_epochs instead (every epoch is scored).")
    fit_df, val_df = train_test_split(train_df, test_size=val_size, random_state=CONFIG["seed"],
                                      shuffle=True, stratify=train_df["LABEL_COLUMN"])
    combos = expand_grid(grid, n_trials)
    print(f"searching {len(combos)} configs x {max_epochs} epochs on {len(fit_df)} fit / {len(val_df)} validation rows")

    rows = []
    for i, combo in enumerate(combos, 1):
        trial_config = {**CONFIG, **combo, "epochs": max_epochs}
        model, history = fine_tune(fit_df["DATA_COLUMN"], fit_df["LABEL_COLUMN"], trial_config,
                                   val_df["DATA_COLUMN"], val_df["LABEL_COLUMN"].tolist(), verbose=False)
        del model
        gc.collect()
        torch.cuda.empty_cache()   # free GPU memory before the next trial

        for record in history:
            rows.append({**combo, "epochs": record["epoch"], "accuracy": record["accuracy"], metric: record[metric]})
        best_epoch = max(history, key=lambda r: r[metric])
        print(f"[{i}/{len(combos)}] {combo}: best {metric} {best_epoch[metric]:.4f} at epoch {best_epoch['epoch']}")

    results = pd.DataFrame(rows).sort_values([metric, "epochs"], ascending=[False, True]).reset_index(drop=True)
    # Read values column by column (a whole row would upcast ints like batch_size to floats).
    best_config = {"epochs": int(results["epochs"].iloc[0])}
    for key in grid:
        value = results[key].iloc[0]
        best_config[key] = value.item() if hasattr(value, "item") else value
    print(f"\nbest config: {best_config} (validation {metric} {results[metric].iloc[0]:.4f})")
    return best_config, results

search_results = None
if RUN_SEARCH:
    best_config, search_results = search_hyperparameters(train_df, n_trials=N_TRIALS)
    CONFIG.update(best_config)

print("training config:", {k: CONFIG[k] for k in ["lr", "batch_size", "epochs", "weight_decay", "warmup_ratio"]})

In [ ]:
# Search results: the top configs, and validation macro-F1 by epoch for each config.
if search_results is not None:
    display(search_results.head(10).round({"accuracy": 4, "macro_f1": 4}))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for (lr, bs), group in search_results.sort_values("epochs").groupby(["lr", "batch_size"]):
        ax.plot(group["epochs"], group["macro_f1"], marker="o", label=f"lr={lr:g}, batch={bs}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("validation macro-F1")
    ax.set_xticks(range(1, MAX_EPOCHS + 1))
    ax.set_title("Hyperparameter search")
    ax.legend(fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Search skipped (RUN_SEARCH = False).")

## 5. Train the final model and evaluate on the test set

Trains on the full training set with `CONFIG` (the search winner, if you ran it), then evaluates on the test set
**once**.

In [ ]:
model, history = fine_tune(train_df["DATA_COLUMN"], y_train, CONFIG)

bert_pred = predict(model, test_df["DATA_COLUMN"])
results["bert"] = compute_metrics(y_test, bert_pred)

summary = pd.DataFrame({
    name: {"accuracy": r["accuracy"], "macro-F1": r["macro_f1"], **{f"F1 {k}": v for k, v in r["per_class_f1"].items()}}
    for name, r in results.items()
}).T.round(3)
summary

In [ ]:
print(metrics.classification_report(y_test, bert_pred, target_names=LABELS, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, pred) in zip(axes, [("TF-IDF + LogReg", tfidf_pred), ("BERT", bert_pred)]):
    metrics.ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=LABELS, normalize="true", values_format=".2f",
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(f"{name} (rows sum to 1)")
plt.tight_layout()
plt.show()

### Accuracy by source

The four sources were labeled differently (`sentiment.csv` has no neutral class), so check whether the model does
much better on some sources than others.

In [ ]:
by_source = (
    test_df.assign(correct=np.array(bert_pred) == y_test.values)
    .groupby("source")
    .agg(rows=("correct", "size"), bert_accuracy=("correct", "mean"))
    .round(3)
)
by_source

### Error analysis

Read a sample of mistakes: sarcasm, mixed opinions and label noise usually show up here, and they make good
material for the README's limitations section.

In [ ]:
errors = test_df.assign(predicted=[LABELS[p] for p in bert_pred], actual=y_test.map(dict(enumerate(LABELS))))
errors = errors[errors["predicted"] != errors["actual"]]
print(f"{len(errors)} errors out of {len(test_df)}")

pd.set_option("display.max_colwidth", 200)
errors.sample(min(12, len(errors)), random_state=CONFIG["seed"])[["source", "actual", "predicted", "DATA_COLUMN"]]

## 6. Save metrics and the model

Tip: if you mounted Google Drive, copy the model there (`!cp -r model /content/drive/MyDrive/`) so it survives a
runtime disconnect.

In [ ]:
results["config"] = {**CONFIG, "searched": search_results is not None, "train_rows": len(train_df),
                     "test_rows": len(test_df), "transformers": transformers.__version__, "torch": torch.__version__}
results["history"] = history
if search_results is not None:
    results["search_top5"] = search_results.head(5).to_dict(orient="records")
    search_results.to_csv("search_results.csv", index=False)
with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2)

model.save_pretrained("model")
tokenizer.save_pretrained("model")
print("saved metrics.json and model/")

try:
    from google.colab import files
    files.download("metrics.json")
except ImportError:
    pass

## 7. (Optional) Publish to the Hugging Face Hub

The API (`deployment/api`) downloads the model from the Hub. Running this cell asks for a Hugging Face access
token with **write** permission (huggingface.co → Settings → Access Tokens).

In [ ]:
HUB_REPO = "baz08/crypto-sentiment-bert"   # the API's default MODEL_REPO
PUSH = False                                # set to True to upload

if PUSH:
    from huggingface_hub import notebook_login
    notebook_login()
    model.push_to_hub(HUB_REPO)
    tokenizer.push_to_hub(HUB_REPO)
    print(f"https://huggingface.co/{HUB_REPO}")

## 8. Try it

Runs new text through the same cleaning the API uses, then the fine-tuned model.

In [ ]:
"""Light text cleaning applied to every comment before it reaches BERT.

This is the cleaning used for BOTH training (``merge_clean.py``) and
inference (``deployment/api/ML/textprep.py`` is an identical copy, kept in
sync by ``tests/test_textprep.py``). Using the same function in both places
avoids train/serve skew: the model sees the same kind of text in production
that it was trained on.

It is deliberately much lighter than ``cleaner.normalize_corpus``. BERT was
pretrained on natural text, so stopwords, punctuation, and casing all carry
signal ("not good" vs "good", "!!!"), and ``bert-base-uncased`` lowercases
on its own. We only remove things that are pure noise: URLs, HTML entities
left over from Reddit's markdown, encoding artifacts, and extra whitespace.

Standard library only, so it can be imported by the API container and the
data pipeline without extra dependencies.
"""

import html
import re

URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
MARKDOWN_HEADER_PATTERN = re.compile(r"^#{1,6}\s*", flags=re.MULTILINE)
WHITESPACE_PATTERN = re.compile(r"\s+")

# Characters that show up when UTF-8 text was decoded as cp1252/latin-1
# (e.g. "isnâ€™t" instead of "isn't").
MOJIBAKE_MARKERS = ("â€", "Ã", "Â")


def fix_mojibake(text):
    """Best-effort repair of UTF-8 text that was mis-decoded as cp1252."""
    if not any(marker in text for marker in MOJIBAKE_MARKERS):
        return text
    try:
        # cp1252 leaves five byte values undefined (0x81, 0x8D, 0x8F, 0x90,
        # 0x9D); those survive mis-decoding as raw control characters, so
        # fall back to latin-1 for them byte-by-byte.
        raw = b"".join(_to_byte(ch) for ch in text)
        return raw.decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text  # not actually mojibake (or mixed); leave it alone


def _to_byte(ch):
    try:
        return ch.encode("cp1252")
    except UnicodeEncodeError:
        return ch.encode("latin-1")


def clean_for_model(text):
    """Normalize a single comment for BERT. Returns '' for missing input."""
    if not isinstance(text, str):
        return ""
    text = fix_mojibake(text)
    # Reddit exports are sometimes double-escaped ("&amp;#x200B;"), so unescape twice.
    text = html.unescape(html.unescape(text))
    text = text.replace("​", " ")
    text = URL_PATTERN.sub(" ", text)
    text = MARKDOWN_HEADER_PATTERN.sub("", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()

In [ ]:
samples = [
    "I love bitcoin, this dip is a gift",
    "ETH gas fees are absolutely killing me",
    "What wallet do you all use for cold storage?",
    "Great, another rug pull. Exactly what I needed today.",
]
probs = []
model.eval()
with torch.no_grad():
    for text in samples:
        enc = tokenizer(clean_for_model(text), truncation=True, max_length=CONFIG["max_length"], return_tensors="pt").to(device)
        probs.append(torch.softmax(model(**enc).logits, dim=-1)[0].cpu().numpy())

pd.DataFrame(probs, columns=LABELS, index=samples).round(3).assign(prediction=lambda d: d[LABELS].idxmax(axis=1))